# Adaptive Early-Exit GNN for Link Prediction

This notebook reproduces all experiments, results, and figures from:

> *Adaptive Early-Exit Graph Neural Networks for Subgraph-Based Link Prediction*  
> Iva Jurkovic, University of Cambridge, L65 Geometric Deep Learning, 2025-2026

## Structure
1. Environment setup (Colab-specific)
2. Imports and device setup
3. HeaRT dataset download
4. Hyperparameters and seed
5. Data loading
6. Preprocessing (negatives + subgraph caching)
7. Model initialisation
8. Training loop
9. Final test evaluation (500 negatives)
10. Diagnostics (alpha distribution, negative difficulty)
11. Plots and LaTeX table

**Runtime estimate:** ~5h on Colab T4 (training) + ~35 min (test eval).

## 1. Environment Setup

Install compatible PyTorch + PyTorch Geometric wheels for Colab (Python 3.12, CUDA 12.1).

In [8]:
# Mount Google Drive for checkpoint and results persistence
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/L65_Early_Exit'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'✓ Project folder: {PROJECT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Project folder: /content/drive/MyDrive/L65_Early_Exit


In [2]:
# Remove conflicting installs, then install matching Torch + PyG wheels
!pip -q uninstall -y torch torchvision torchaudio torch-geometric pyg-lib \
  torch-scatter torch-sparse torch-cluster torch-spline-conv

!pip -q install --no-cache-dir \
  torch==2.2.2+cu121 torchvision==0.17.2+cu121 torchaudio==2.2.2+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

!pip -q install --no-cache-dir \
  pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv \
  -f https://data.pyg.org/whl/torch-2.2.2+cu121.html

!pip -q install --no-cache-dir torch-geometric ogb
!pip -q install 'numpy<2' --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 313.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 369.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 223.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 274.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 337.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 337.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 333.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 337.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 354.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 339.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 276.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 274.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# Verify installation
import torch, sys, platform
import torch_geometric, torch_scatter, torch_sparse
import numpy as np

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('PyG:', torch_geometric.__version__)
print('NumPy:', np.__version__)

# Quick scatter sanity check
from torch_scatter import scatter_add
x = torch.tensor([1., 2., 3., 4.], device='cuda')
idx = torch.tensor([0, 0, 1, 1], device='cuda')
assert scatter_add(x, idx, dim=0).tolist() == [3., 7.], 'scatter_add failed'
print('✓ All checks passed')

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.2.2+cu121
CUDA available: True
GPU: Tesla T4
PyG: 2.7.0
NumPy: 1.26.4
✓ All checks passed


## 2. Imports and Device

In [12]:
import sys, os

# Add code folder to path so module imports work from notebook root
sys.path.insert(0, '/content/drive/MyDrive/L65_Repo')

import torch
import numpy as np
import json
import matplotlib.pyplot as plt
import pandas as pd

from model import AdaptiveSAGE
from data_utils import (
    load_cora_with_heart,
    precompute_heart_negatives_fast,
    precompute_all_subgraphs,
    build_train_edge_list,
    build_val_edge_list,
)
from train_eval import (
    train_one_epoch_cached,
    evaluate_heart,
    evaluate_test_500,
    analyze_exit_distribution,
)
from visualisation import (
    plot_training_curves,
    plot_baseline_comparison,
    plot_efficiency_comparison,
    plot_exit_distribution_enhanced,
    print_latex_table,
)
from utils import set_seed, save_results, load_results, print_test_results

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 3. HeaRT Dataset Download

In [13]:
!git clone https://github.com/Juanhui28/HeaRT.git
%cd HeaRT
!bash download_data.sh

Cloning into 'HeaRT'...
remote: Enumerating objects: 1003, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 1003 (delta 37), reused 90 (delta 36), pack-reused 894 (from 1)
Receiving objects: 100% (1003/1003), 831.86 KiB | 5.30 MiB/s, done.
Resolving deltas: 100% (532/532), done.
/content/HeaRT
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0  838M    0 55943    0     0  38456      0  6:21:10  0:00:01  6:21:09 38448dataset/
dataset/citeseer/
dataset/citeseer/heart_test_samples.npy
dataset/citeseer/test_pos.txt
dataset/citeseer/train_neg.txt
dataset/citeseer/test_neg.txt
dataset/citeseer/train_pos.txt
dataset/citeseer/gnn_feature
dataset/citeseer/heart_valid_samples.npy
dataset/citeseer/valid_pos.txt
dataset/citeseer/valid_neg.txt
dataset/ogbl-citation2/
dataset/ogbl-citation2/heart_valid_samples.npy
 10  838M

In [14]:
%cd /content/HeaRT
!pwd

/content/HeaRT
/content/HeaRT


## 4. Hyperparameters and Seed

All hyperparameters are defined here. No tuning was performed; values reflect
architectural feasibility exploration as described in Section 4.2 of the report.

In [15]:
SEED = 123
set_seed(SEED)

args = {
    'hidden_dim': 128,       # Node embedding dimensionality
    'L_max': 3,              # Maximum propagation depth
    'dropout': 0.1,          # Dropout after each SAGE layer
    'lr': 0.001,             # Adam learning rate
    'l2': 0.001,             # Adam weight decay
    'lambda_depth': 0.0,     # Depth penalty coefficient (0 = no explicit regularisation)
    'batch_size': 8,         # Edges per gradient update
    'num_neg_train': 5,      # Hard negatives per positive during training
    'epochs': 50,
    'eval_every': 10,        # Validation frequency (epochs)
    'eval_subset': 100,      # Edges to evaluate during training (None = all)
}

print('Hyperparameters:')
for k, v in args.items():
    print(f'  {k}: {v}')

✓ Random seed set to 123
Hyperparameters:
  hidden_dim: 128
  L_max: 3
  dropout: 0.1
  lr: 0.001
  l2: 0.001
  lambda_depth: 0.0
  batch_size: 8
  num_neg_train: 5
  epochs: 50
  eval_every: 10
  eval_subset: 100


## 5. Data Loading

In [16]:
DATA_PATH = 'dataset/cora'
data, train_pos, val_pos, test_pos, val_neg, test_neg = load_cora_with_heart(DATA_PATH, device)

Processing...


Graph: 2708 nodes, 10556 edges
Features: torch.Size([2708, 1433])
val negatives: torch.Size([263, 500, 2]) (HeaRT hard negatives)
test negatives: torch.Size([527, 500, 2]) (HeaRT hard negatives)

Split sizes:
  Train: 4488 edges
  Val:   263 edges
  Test:  527 edges


Done!


## 6. Preprocessing

### 6a. Training Negatives (HeaRT-style, Common Neighbour scoring)

In [17]:
train_negatives = precompute_heart_negatives_fast(
    train_pos, data, K=args['num_neg_train']
)
print(f'✓ {len(train_negatives)} positive edges, each with {len(train_negatives[0])} hard negatives')

Generating training negatives: 100%|██████████| 4488/4488 [00:13<00:00, 329.03it/s]

✓ 4488 positive edges, each with 5 hard negatives


### 6b. SEAL-Style Subgraph Caching

Subgraphs are precomputed at all depths 1..L_max for all training and validation
edges to avoid redundant extraction during training (11x speedup over on-the-fly).

In [18]:
# Build deduplicated list of all training edges (positives + negatives)
all_train_edges = build_train_edge_list(train_pos, train_negatives)
print(f'Total unique training edges: {len(all_train_edges):,}')

# Cache subgraphs (stored on CPU, moved to GPU in forward pass)
cached_subgraphs = precompute_all_subgraphs(
    all_train_edges, data, L_max=args['L_max'], device='cpu'
)
print('✓ Training subgraphs cached')

Building edge list: 100%|███████████████| 4488/4488 [00:00<00:00, 200963.35it/s]


Total unique training edges to cache: 12,591
Total unique training edges: 12,591

Caching subgraphs for 12,591 edges at depths 1-3


Caching: 100%|███████████████████████████| 12591/12591 [00:19<00:00, 632.99it/s]


✓ Caching complete!
  Total subgraphs: 37,773 (12,591 edges × 3 depths)
  Avg nodes/subgraph: ~143
  Est. memory: ~82 MB
  Time: 0.3 min  (633 edges/sec)

✓ Training subgraphs cached


In [19]:
# Cache validation subgraphs
val_edges_list = build_val_edge_list(val_pos, val_neg)
print(f'Caching {len(val_edges_list)} validation edges...')
val_cached_subgraphs = precompute_all_subgraphs(
    val_edges_list, data, L_max=args['L_max']
)
print('✓ Validation subgraphs cached')

Caching 131763 validation edges...

Caching subgraphs for 131,763 edges at depths 1-3


Caching: 100%|█████████████████████████| 131763/131763 [03:48<00:00, 577.15it/s]


✓ Caching complete!
  Total subgraphs: 395,289 (131,763 edges × 3 depths)
  Avg nodes/subgraph: ~79
  Est. memory: ~476 MB
  Time: 3.8 min  (577 edges/sec)

✓ Validation subgraphs cached


### 6c. Cache Verification

In [ ]:
# Verify cache is populated correctly
u, v = train_pos[0].tolist()
print(f'Test edge: ({u}, {v})')
for depth in range(1, args['L_max'] + 1):
    sg = cached_subgraphs[(u, v)][depth]
    print(f'  Depth {depth}: {sg["num_nodes"]} nodes, {sg["edge_index"].size(1)} edges')
print('✓ Cache verification passed')

## 7. Model Initialisation

In [ ]:
model = AdaptiveSAGE(
    in_dim=data.num_features,
    hidden_dim=args['hidden_dim'],
    L_max=args['L_max'],
    dropout=args['dropout'],
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=args['lr'], weight_decay=args['l2']
)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Device: {device}')

# Smoke test
model.eval()
with torch.no_grad():
    score, depth, alpha = model.forward_pair_train_cached(
        cached_subgraphs, u, v, data.x
    )
print(f'✓ Forward pass: score={score.item():.4f}, depth={depth.item():.2f}, alpha={alpha.detach().cpu().numpy()}')

## 8. Training

Training uses crash-proof saving: best model and full checkpoint saved to Google Drive
every validation epoch and every 10 epochs respectively.

In [ ]:
MODEL_PATH = f'{PROJECT_DIR}/best_model.pt'
RESULTS_PATH = f'{PROJECT_DIR}/results_log.json'
CHECKPOINT_DIR = f'{PROJECT_DIR}/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

results_log = {
    'seed': SEED,
    'hyperparameters': args.copy(),
    'training_history': [],
    'final_results': {},
}

best_val_mrr = 0.0

print('=' * 70)
print('TRAINING - saving best model and checkpoints to Google Drive')
print('=' * 70)

for epoch in range(args['epochs']):

    train_stats = train_one_epoch_cached(
        model, data, train_pos, train_negatives,
        cached_subgraphs, args, optimizer, device
    )

    print(f"Epoch {epoch+1:03d}/{args['epochs']} | "
          f"Loss: {train_stats['loss']:.4f} | "
          f"Rank Loss: {train_stats['rank_loss']:.4f} | "
          f"Depth Pen: {train_stats['depth_pen']:.4f} | "
          f"Avg Depth: {train_stats['avg_depth']:.2f}")

    # Validation every eval_every epochs
    if (epoch + 1) % args['eval_every'] == 0:
        print('\n' + '-' * 60)
        print('Evaluating...')

        val_stats = evaluate_heart(
            model, data, val_pos, val_neg,
            val_cached_subgraphs, args, mode='val'
        )

        results_log['training_history'].append({
            'epoch': epoch + 1,
            'train_loss': train_stats['loss'],
            'train_rank_loss': train_stats['rank_loss'],
            'train_depth_pen': train_stats['depth_pen'],
            'train_avg_depth': train_stats['avg_depth'],
            'val_mrr': val_stats['mrr'],
            'val_hits10': val_stats['hits@10'],
            'val_hits20': val_stats['hits@20'],
            'val_hits50': val_stats['hits@50'],
            'val_avg_depth': val_stats['avg_depth'],
            'computation_saved': val_stats['computation_saved'],
        })

        save_results(results_log, RESULTS_PATH)

        print(f"\nEpoch {epoch+1:03d}/{args['epochs']}")
        print(f"  Train: Loss={train_stats['loss']:.4f}, AvgDepth={train_stats['avg_depth']:.2f}")
        print(f"  Val:   MRR={val_stats['mrr']:.4f}, Hits@20={val_stats['hits@20']:.4f}, AvgDepth={val_stats['avg_depth']:.2f}")

        if val_stats['mrr'] > best_val_mrr:
            best_val_mrr = val_stats['mrr']
            torch.save(model.state_dict(), MODEL_PATH)
            print(f'  ✓✓✓ NEW BEST MODEL! Saved (MRR={best_val_mrr:.4f})')

    # Full checkpoint every 10 epochs
    if epoch % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_mrr': best_val_mrr,
            'results_log': results_log,
        }, f'{CHECKPOINT_DIR}/checkpoint_epoch{epoch}.pt')
        print(f'  ✓ Checkpoint saved (epoch {epoch})')

# Final save
torch.save(model.state_dict(), f'{PROJECT_DIR}/final_model_epoch{args["epochs"]}.pt')
torch.save({
    'epoch': args['epochs'],
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_val_mrr': best_val_mrr,
    'results_log': results_log,
    'args': args,
}, f'{PROJECT_DIR}/final_complete_state.pt')
save_results(results_log, RESULTS_PATH)

print('\n' + '=' * 70)
print('TRAINING COMPLETE')
print(f'  Best validation MRR: {best_val_mrr:.4f}')
print('=' * 70)

## 9. Final Test Evaluation

Load best checkpoint and evaluate against all 500 HeaRT hard negatives.
Expected runtime: 30–45 minutes on Colab T4.

In [ ]:
# Load best model
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
print(f'✓ Loaded best model from {MODEL_PATH}')

# Run full test evaluation
test_results = evaluate_test_500(model, data, test_pos, test_neg, device)

# Print and save
print_test_results(test_results)
save_results(test_results, f'{PROJECT_DIR}/test_results_FINAL_500negs.json')

# Consistency check against validation
results_log = load_results(RESULTS_PATH)
val_mrr = results_log['training_history'][-1]['val_mrr']
print(f'\nVal MRR:  {val_mrr:.4f}')
print(f'Test MRR: {test_results["mrr"]:.4f}')
print(f'Diff:     {(test_results["mrr"] - val_mrr)*100:+.2f}%')
if abs(test_results['mrr'] - val_mrr) < 0.03:
    print('✓✓✓ Results consistent — no val/test leakage')
else:
    print('!!!! Large val/test gap — review before reporting')

## 10. Diagnostics

### 10a. Halting Weight (Alpha) Distribution

Verify the ACT mechanism is non-degenerate (not collapsing to depth 1).

In [ ]:
print('Checking alpha distributions...')

model.eval()
alphas_list = []

with torch.no_grad():
    for i in range(20):
        u_i, v_i = train_pos[i].tolist()
        score, depth, alpha = model.forward_pair_train_cached(
            cached_subgraphs, u_i, v_i, data.x
        )
        alphas_list.append(alpha.cpu().numpy())
        if i < 5:
            print(f'  Edge {i}: alpha={alpha.cpu().numpy()}, depth={depth.item():.2f}')

avg_alpha = np.mean(alphas_list, axis=0)
print(f'\nAverage alpha across 20 edges: {avg_alpha}')
print('Non-degenerate if distributed across layers (not [~1, ~0, ~0])')

### 10b. Exit Layer Distribution

In [ ]:
exit_dist = analyze_exit_distribution(model, data, val_pos, val_cached_subgraphs)
print(f'Exit distribution: {exit_dist}')

results_log['final_results']['exit_distribution'] = exit_dist
save_results(results_log, RESULTS_PATH)

### 10c. HeaRT Negative Difficulty Verification

Verify negatives are genuinely hard (CN > 2 indicates structural difficulty).

In [ ]:
from collections import defaultdict

adj_list = defaultdict(set)
for i in range(data.edge_index.size(1)):
    src, dst = data.edge_index[0, i].item(), data.edge_index[1, i].item()
    adj_list[src].add(dst)
    adj_list[dst].add(src)

u0, v0 = train_pos[0].tolist()
negatives_sample = train_negatives[0]
u_neighbors = adj_list[u0]

print(f'Positive edge: ({u0}, {v0})')
print(f'u has {len(u_neighbors)} neighbors')

cn_scores = []
for u_neg, v_neg in negatives_sample[:10]:
    if u_neg == u0:
        cn = len(u_neighbors & adj_list[v_neg])
    else:
        cn = len(adj_list[u_neg] & adj_list[v0])
    cn_scores.append(cn)
    print(f'  Negative ({u_neg}, {v_neg}): CN = {cn}')

print(f'\nAverage CN of negatives: {np.mean(cn_scores):.2f}')
print('CN > 2: Negatives are hard ✅' if np.mean(cn_scores) > 2 else 'CN < 1: Negatives are too easy ❌')

## 11. Figures and LaTeX Table

Reproduces Figures 1–3 from the report and the LaTeX results table.

In [ ]:
# Subset of results needed for plotting
test_results_final = {
    'mrr': test_results['mrr'],
    'hits@20': test_results['hits@20'],
    'avg_depth': test_results['avg_depth'],
}

# Figure 1: Training curves
plot_training_curves(results_log, f'{PROJECT_DIR}/training_curves.png')

# Figure 2: Baseline comparison
plot_baseline_comparison(test_results_final, f'{PROJECT_DIR}/baseline_comparison_FINAL.png')

# Figure 3: Efficiency trade-off
plot_efficiency_comparison(test_results_final, f'{PROJECT_DIR}/efficiency_comparison_FINAL.png')

# Figure 4: Exit distribution
plot_exit_distribution_enhanced(exit_dist, f'{PROJECT_DIR}/exit_distribution.png')

# LaTeX table
print_latex_table(test_results_final)

print('\n✓ All outputs generated and saved to Google Drive')